# PitchLens — préparation du dataset `v001`

Ce notebook transforme l’audit d’Infactory en un premier dataset propre et reproductible. Il ne modifie jamais `data/raw`.

Objectifs :

1. classer chaque frame selon la qualité de son annotation ;
2. isoler les cas qui nécessitent une décision humaine ;
3. créer train/validation/test par **séquence complète** ;
4. matérialiser `data/processed/v001` ;
5. vérifier automatiquement le résultat.

## 1. Politique de nettoyage

Pour cette première version, nous préférons perdre quelques images douteuses plutôt que d’apprendre au modèle des erreurs.

| État | Règle | Décision `v001` |
|---|---|---|
| `positive_single` | exactement une boîte valide | incluse |
| `negative` | aucun fichier label ou label vide | incluse avec un label vide |
| `review_invalid` | ligne mal formée ou boîte de surface nulle | exclue, file de revue |
| `review_multi` | plusieurs boîtes ballon valides | exclue, file de revue |

Cette règle est volontairement conservatrice. Une future `v002` pourra réintégrer les cas corrigés sans modifier `v001`.

In [ ]:
from collections import Counter, defaultdict
from pathlib import Path
import csv
import ctypes
import json
import os
import random
import shutil
import sys

import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from PIL import Image, ImageDraw

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from pitchlens.data.audit_yolo import audit

SOURCE_ROOT = PROJECT_ROOT / "data/raw/infactory_ball_tracking/data"
PROCESSED_ROOT = PROJECT_ROOT / "data/processed/v001"
SOURCE_MANIFEST = PROJECT_ROOT / "data/manifests/infactory_ball_tracking.json"
DERIVED_MANIFEST = PROJECT_ROOT / "data/manifests/infactory_v001.json"
REPORT_PATH = PROJECT_ROOT / "models/reports/infactory_v001_summary.json"

if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(f"Dataset Infactory absent : {SOURCE_ROOT}")

print("Source :", SOURCE_ROOT)
print("Destination :", PROCESSED_ROOT)

## 2. Classer toutes les frames

Nous relisons les annotations depuis la source. Le nom avant le dernier `_` est l’UUID de séquence ; le nombre final est l’index temporel de la frame.

In [ ]:
def inspect_label(image_path):
    label_path = image_path.with_suffix(".txt")
    if not label_path.exists():
        return "negative", 0, []

    rows = [line.strip() for line in label_path.read_text().splitlines() if line.strip()]
    if not rows:
        return "negative", 0, []

    valid_boxes = []
    invalid_rows = 0
    for row in rows:
        fields = row.split()
        if len(fields) != 5:
            invalid_rows += 1
            continue
        try:
            class_id = int(fields[0])
            x, y, width, height = map(float, fields[1:])
        except ValueError:
            invalid_rows += 1
            continue
        if (
            class_id != 0
            or not all(0 <= value <= 1 for value in (x, y, width, height))
            or width == 0
            or height == 0
        ):
            invalid_rows += 1
            continue
        valid_boxes.append((x, y, width, height))

    if invalid_rows:
        return "review_invalid", len(valid_boxes), valid_boxes
    if len(valid_boxes) == 1:
        return "positive_single", 1, valid_boxes
    if len(valid_boxes) > 1:
        return "review_multi", len(valid_boxes), valid_boxes
    return "negative", 0, []

records = []
for image_path in sorted(SOURCE_ROOT.rglob("*.jpg")):
    sequence, frame_text = image_path.stem.rsplit("_", 1)
    status, box_count, boxes = inspect_label(image_path)
    records.append({
        "sequence": sequence,
        "frame": int(frame_text),
        "status": status,
        "box_count": box_count,
        "image_path": image_path,
        "label_path": image_path.with_suffix(".txt"),
        "boxes": boxes,
    })

status_counts = Counter(record["status"] for record in records)
accepted_statuses = {"positive_single", "negative"}
accepted_count = sum(status_counts[status] for status in accepted_statuses)
review_count = len(records) - accepted_count

display(Markdown(f"""
| État | Images | Décision |
|---|---:|---|
| Une boîte valide | {status_counts['positive_single']:,} | inclure |
| Négatif | {status_counts['negative']:,} | inclure |
| Annotation invalide | {status_counts['review_invalid']:,} | revue |
| Plusieurs boîtes | {status_counts['review_multi']:,} | revue |
| **Total accepté** | **{accepted_count:,}** | `v001` |
| **Total en revue** | **{review_count:,}** | hors `v001` |
"""))

assert len(records) == 7152
assert accepted_count == 7014
assert review_count == 138

### Inspection de la file de revue

La galerie prend quatre annotations invalides et quatre images multi-boîtes. Les boîtes valides sont rouges ; une image invalide sans rectangle possède uniquement une boîte de surface nulle. Cette revue explique pourquoi ces frames sont exclues plutôt que corrigées automatiquement.

In [ ]:
def draw_record(record):
    image = Image.open(record["image_path"]).convert("RGB")
    drawing = ImageDraw.Draw(image)
    for x, y, width, height in record["boxes"]:
        left = (x - width / 2) * image.width
        top = (y - height / 2) * image.height
        right = (x + width / 2) * image.width
        bottom = (y + height / 2) * image.height
        drawing.rectangle((left, top, right, bottom), outline="red", width=4)
    return image

def one_example_per_sequence(status, limit=4):
    examples_by_sequence = {}
    for record in records:
        if record["status"] == status:
            examples_by_sequence.setdefault(record["sequence"], record)
    return list(examples_by_sequence.values())[:limit]

review_examples = (
    one_example_per_sequence("review_invalid")
    + one_example_per_sequence("review_multi")
)

fig, axes = plt.subplots(2, 4, figsize=(17, 8))
for axis, record in zip(axes.flat, review_examples):
    axis.imshow(draw_record(record))
    axis.set_title(f"{record['status']} — {record['box_count']} boîte(s) valide(s)")
    axis.axis("off")
plt.tight_layout()
plt.show()

## 3. Construire les splits par séquence

Nous visons 70 % train, 15 % validation et 15 % test. Comme les 54 séquences ont des longueurs très différentes, prendre simplement 70 % des UUID ne suffit pas.

L’algorithme teste 25 000 répartitions déterministes et conserve celle qui équilibre le mieux :

- le nombre total de frames acceptées ;
- les frames positives ;
- les frames négatives.

Chaque UUID reste intégralement dans un seul split.

In [ ]:
sequence_stats = defaultdict(Counter)
for record in records:
    stats = sequence_stats[record["sequence"]]
    stats["all"] += 1
    stats[record["status"]] += 1
    if record["status"] in accepted_statuses:
        stats["accepted"] += 1

sequence_ids = sorted(sequence_stats)
split_sequence_counts = {"train": 38, "val": 8, "test": 8}
target_ratios = {"train": 0.70, "val": 0.15, "test": 0.15}
metrics = ("accepted", "positive_single", "negative")
metric_totals = {
    metric: sum(stats[metric] for stats in sequence_stats.values())
    for metric in metrics
}

def candidate_mapping(order):
    train_end = split_sequence_counts["train"]
    val_end = train_end + split_sequence_counts["val"]
    return {
        **{sequence: "train" for sequence in order[:train_end]},
        **{sequence: "val" for sequence in order[train_end:val_end]},
        **{sequence: "test" for sequence in order[val_end:]},
    }

def split_summary(mapping):
    summary = {split: Counter() for split in target_ratios}
    for sequence, split in mapping.items():
        summary[split]["sequences"] += 1
        summary[split].update(sequence_stats[sequence])
    return summary

def balance_score(mapping):
    summary = split_summary(mapping)
    return sum(
        (summary[split][metric] / metric_totals[metric] - target_ratios[split]) ** 2
        for split in target_ratios
        for metric in metrics
    )

rng = random.Random(42)
best_mapping = None
best_score = float("inf")
for _ in range(25_000):
    order = sequence_ids.copy()
    rng.shuffle(order)
    mapping = candidate_mapping(order)
    score = balance_score(mapping)
    if score < best_score:
        best_mapping = mapping
        best_score = score

for record in records:
    record["split"] = best_mapping[record["sequence"]]

summary = split_summary(best_mapping)
rows = [
    "| Split | Séquences | Acceptées | Positives | Négatives | En revue |",
    "|---|---:|---:|---:|---:|---:|",
]
for split in ("train", "val", "test"):
    stats = summary[split]
    review = stats["review_invalid"] + stats["review_multi"]
    rows.append(
        f"| {split} | {stats['sequences']} | {stats['accepted']:,} "
        f"({stats['accepted'] / accepted_count:.1%}) | {stats['positive_single']:,} | "
        f"{stats['negative']:,} | {review:,} |"
    )
display(Markdown("\n".join(rows)))

assert len(best_mapping) == 54
assert set(best_mapping.values()) == {"train", "val", "test"}
assert sum(stats["accepted"] for stats in summary.values()) == accepted_count

### Contrôle visuel de l’équilibre

Le graphique compare la cible théorique aux proportions réellement obtenues. Une correspondance exacte est impossible avec des séquences indivisibles, mais l’écart doit rester faible.

In [ ]:
splits = ["train", "val", "test"]
positions = list(range(len(splits)))
actual_accepted = [summary[split]["accepted"] / accepted_count for split in splits]
actual_positive = [
    summary[split]["positive_single"] / status_counts["positive_single"]
    for split in splits
]
targets = [target_ratios[split] for split in splits]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar([x - 0.25 for x in positions], targets, 0.25, label="cible")
ax.bar(positions, actual_accepted, 0.25, label="frames acceptées")
ax.bar([x + 0.25 for x in positions], actual_positive, 0.25, label="positifs")
ax.set_xticks(positions, splits)
ax.set_ylabel("Part du dataset")
ax.set_title("Équilibre obtenu sans couper les séquences")
ax.legend()
plt.show()

## 4. Matérialiser `data/processed/v001`

Les images acceptées sont créées sous forme de **clones APFS** : elles ont un inode indépendant mais partagent initialement les mêmes blocs physiques. Cela évite de recopier environ 1 Go tout en protégeant `raw` si une copie est modifiée plus tard. Sur un autre système de fichiers, le notebook utilise une copie classique.

Les petits fichiers labels positifs sont copiés normalement. Pour un négatif, nous créons un fichier `.txt` vide, convention comprise par YOLO. Les 138 cas douteux sont uniquement inscrits dans `review_queue.csv`.

La cellule est idempotente : si une `v001` complète existe déjà, elle ne l’écrase pas.

In [ ]:
def relative_to_project(path):
    return str(path.relative_to(PROJECT_ROOT))

if sys.platform == "darwin":
    clonefile = ctypes.CDLL("libSystem.B.dylib", use_errno=True).clonefile
    clonefile.argtypes = [ctypes.c_char_p, ctypes.c_char_p, ctypes.c_int]
    clonefile.restype = ctypes.c_int
else:
    clonefile = None

def clone_or_copy(source, destination):
    if clonefile is None:
        shutil.copy2(source, destination)
        return
    result = clonefile(os.fsencode(source), os.fsencode(destination), 0)
    if result != 0:
        error_number = ctypes.get_errno()
        raise OSError(error_number, os.strerror(error_number), destination)

manifest_fields = [
    "sequence", "frame", "split", "status", "box_count",
    "source_image", "source_label", "output_image", "output_label",
]

if PROCESSED_ROOT.exists() and not (PROCESSED_ROOT / "manifest.csv").exists():
    raise RuntimeError(
        f"Destination partielle détectée : {PROCESSED_ROOT}. "
        "Inspection manuelle nécessaire ; aucun écrasement automatique."
    )

if not PROCESSED_ROOT.exists():
    for split in ("train", "val", "test"):
        (PROCESSED_ROOT / split / "images").mkdir(parents=True)
        (PROCESSED_ROOT / split / "labels").mkdir(parents=True)

    manifest_rows = []
    for record in records:
        output_image = ""
        output_label = ""
        if record["status"] in accepted_statuses:
            image_destination = (
                PROCESSED_ROOT / record["split"] / "images" / record["image_path"].name
            )
            label_destination = (
                PROCESSED_ROOT / record["split"] / "labels" / record["label_path"].name
            )
            clone_or_copy(record["image_path"], image_destination)
            if record["status"] == "positive_single":
                shutil.copy2(record["label_path"], label_destination)
            else:
                label_destination.write_text("")
            output_image = relative_to_project(image_destination)
            output_label = relative_to_project(label_destination)

        manifest_rows.append({
            "sequence": record["sequence"],
            "frame": record["frame"],
            "split": record["split"],
            "status": record["status"],
            "box_count": record["box_count"],
            "source_image": relative_to_project(record["image_path"]),
            "source_label": (
                relative_to_project(record["label_path"])
                if record["label_path"].exists() else ""
            ),
            "output_image": output_image,
            "output_label": output_label,
        })

    with (PROCESSED_ROOT / "manifest.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=manifest_fields)
        writer.writeheader()
        writer.writerows(manifest_rows)

    with (PROCESSED_ROOT / "review_queue.csv").open("w", newline="") as stream:
        writer = csv.DictWriter(stream, fieldnames=manifest_fields)
        writer.writeheader()
        writer.writerows(
            row for row in manifest_rows if row["status"].startswith("review_")
        )

    (PROCESSED_ROOT / "dataset.yaml").write_text(
        "train: train/images\nval: val/images\ntest: test/images\nnames:\n  0: ball\n"
    )
    print("v001 matérialisée.")
else:
    print("v001 existe déjà : aucun fichier écrasé, passage à la vérification.")

## 5. Enregistrer la provenance et vérifier

Le petit manifeste JSON peut être versionné sur GitHub : il décrit comment reconstruire `v001` sans contenir les images. Le CSV détaillé reste avec le dataset local.

In [ ]:
source_provenance = json.loads(SOURCE_MANIFEST.read_text())
split_counts = {
    split: {
        "sequences": summary[split]["sequences"],
        "images": summary[split]["accepted"],
        "positive_images": summary[split]["positive_single"],
        "negative_images": summary[split]["negative"],
    }
    for split in ("train", "val", "test")
}
derived_manifest = {
    "dataset_id": "pitchlens_infactory_v001",
    "quality_status": "working_dataset_visual_review_required",
    "source_registry_id": source_provenance["registry_id"],
    "source_revision": source_provenance["resolved_revision"],
    "source_url": source_provenance["source_url"],
    "split_unit": "complete UUID sequence",
    "split_target_ratios": target_ratios,
    "split_search_seed": 42,
    "split_search_candidates": 25_000,
    "included_statuses": sorted(accepted_statuses),
    "excluded_statuses": ["review_invalid", "review_multi"],
    "counts": {
        "source_images": len(records),
        "included_images": accepted_count,
        "review_images": review_count,
        "positive_images": status_counts["positive_single"],
        "negative_images": status_counts["negative"],
        "invalid_images": status_counts["review_invalid"],
        "multi_box_images": status_counts["review_multi"],
    },
    "splits": split_counts,
}
rendered_manifest = json.dumps(derived_manifest, indent=2) + "\n"
DERIVED_MANIFEST.write_text(rendered_manifest)
REPORT_PATH.write_text(rendered_manifest)

prepared_audit = audit(PROCESSED_ROOT, target_size=640)
prepared_ball = prepared_audit["class_metrics"][0]

assert prepared_audit["images"] == accepted_count
assert prepared_audit["issues_count"] == 0
assert prepared_ball["boxes"] == status_counts["positive_single"]
assert prepared_ball["multi_box_images_count"] == 0
assert prepared_audit["negative_images"] == status_counts["negative"]

sequence_splits = defaultdict(set)
for record in records:
    sequence_splits[record["sequence"]].add(record["split"])
assert all(len(splits_for_sequence) == 1 for splits_for_sequence in sequence_splits.values())

display(Markdown(f"""
### Vérification réussie

- **{prepared_audit['images']:,} images** dans `v001` ;
- **{prepared_ball['boxes']:,} boîtes ballon**, toutes uniques par image ;
- **{prepared_audit['negative_images']:,} négatifs** ;
- **0 annotation invalide** ;
- **0 UUID partagé entre les splits** ;
- **{review_count} images** conservées dans la file de revue, hors entraînement.
"""))

## 6. Contrôle qualité visuel stratifié

Les vérifications précédentes prouvent que les fichiers sont cohérents, mais pas que chaque boîte encadre réellement un ballon. Nous affichons donc **20 images par split**, toujours sélectionnées avec la graine `42` : quatre exemples pour chacune des quatre tailles de ballon et quatre négatifs.

La taille indiquée est le diamètre équivalent de la boîte après letterbox vers `640 × 640` : `sqrt(largeur × hauteur)`. Ce n'est pas la taille physique du ballon ; c'est une mesure simple de la difficulté pour le modèle.

| Groupe | Diamètre équivalent à 640 | Pourquoi l'inspecter |
|---|---:|---|
| `tiny` | `< 8 px` | cas les plus faciles à perdre au resize |
| `small` | `8–16 px` | cœur du dataset |
| `medium` | `16–32 px` | ballon normalement lisible |
| `large` | `≥ 32 px` | gros plans et changements de caméra |
| `negative` | aucune boîte | vérifier qu'un ballon visible n'a pas été oublié |

Chaque titre rappelle l'origine (`Infactory`), le split, la séquence et la frame. La sélection privilégie des séquences différentes afin d'éviter de regarder vingt images presque identiques. Les galeries sont aussi enregistrées dans `models/reports/`.

In [ ]:
import math

QA_REPORT_PATH = PROJECT_ROOT / "models/reports/infactory_v001_qa.json"
QA_CATEGORY_ORDER = ("tiny", "small", "medium", "large", "negative")
QA_SAMPLES_PER_CATEGORY = 4

def equivalent_diameter_640(record):
    if record["status"] == "negative":
        return None
    with Image.open(record["image_path"]) as image:
        image_width, image_height = image.size
    _, _, box_width, box_height = record["boxes"][0]
    scale = min(640 / image_width, 640 / image_height)
    width_640 = box_width * image_width * scale
    height_640 = box_height * image_height * scale
    return math.sqrt(width_640 * height_640)

def qa_category(record):
    diameter = equivalent_diameter_640(record)
    if diameter is None:
        return "negative"
    if diameter < 8:
        return "tiny"
    if diameter < 16:
        return "small"
    if diameter < 32:
        return "medium"
    return "large"

def diverse_sample(candidates, sample_count, seed):
    # Un exemple par séquence au premier passage, puis remplissage si nécessaire.
    local_rng = random.Random(seed)
    by_sequence = defaultdict(list)
    for candidate in candidates:
        by_sequence[candidate["sequence"]].append(candidate)
    sequences = sorted(by_sequence)
    local_rng.shuffle(sequences)
    selected = []
    for sequence in sequences:
        selected.append(local_rng.choice(by_sequence[sequence]))
        if len(selected) == sample_count:
            return selected
    remaining = [candidate for candidate in candidates if candidate not in selected]
    local_rng.shuffle(remaining)
    return selected + remaining[: sample_count - len(selected)]

accepted_records = [
    record for record in records if record["status"] in accepted_statuses
]
for record in accepted_records:
    record["qa_category"] = qa_category(record)
    record["diameter_640"] = equivalent_diameter_640(record)

qa_population = {
    split: Counter(
        record["qa_category"]
        for record in accepted_records
        if record["split"] == split
    )
    for split in ("train", "val", "test")
}

qa_samples = {}
for split_index, split in enumerate(("train", "val", "test")):
    fig, axes = plt.subplots(5, 4, figsize=(18, 18))
    split_samples = []
    for category_index, category in enumerate(QA_CATEGORY_ORDER):
        candidates = [
            record
            for record in accepted_records
            if record["split"] == split and record["qa_category"] == category
        ]
        selected = diverse_sample(
            candidates, QA_SAMPLES_PER_CATEGORY, 42 + split_index * 10 + category_index
        )
        assert len(selected) == QA_SAMPLES_PER_CATEGORY
        split_samples.extend(selected)
        for column, record in enumerate(selected):
            axis = axes[category_index, column]
            axis.imshow(draw_record(record))
            diameter = record["diameter_640"]
            difficulty = "sans boîte" if diameter is None else f"{diameter:.1f} px @640"
            axis.set_title(
                f"Infactory · {split} · {category}\n"
                f"seq {record['sequence'][:8]}… · frame {record['frame']} · {difficulty}",
                fontsize=9,
            )
            axis.axis("off")
    qa_samples[split] = split_samples
    fig.suptitle(
        f"PitchLens v001 — contrôle visuel {split} — source : Infactory Ball Tracking",
        fontsize=16,
    )
    plt.tight_layout(rect=(0, 0, 1, 0.98))
    gallery_path = PROJECT_ROOT / f"models/reports/infactory_v001_qa_{split}.jpg"
    fig.savefig(gallery_path, dpi=140, bbox_inches="tight")
    plt.show()

population_rows = [
    "| Split | Tiny | Small | Medium | Large | Négatifs |",
    "|---|---:|---:|---:|---:|---:|",
]
for split in ("train", "val", "test"):
    counts = qa_population[split]
    population_rows.append(
        f"| {split} | {counts['tiny']:,} | {counts['small']:,} | "
        f"{counts['medium']:,} | {counts['large']:,} | {counts['negative']:,} |"
    )
display(Markdown("### Population par niveau de difficulté\n\n" + "\n".join(population_rows)))

## 7. Transformer la QA en file de correction

La lecture des 60 images a invalidé l'hypothèse « une boîte valide syntaxiquement = un vrai ballon ». Nous avons vu des boîtes sur un crâne, une zone lumineuse ou la foule, ainsi qu'un gardien tenant un ballon dans une image déclarée négative. **`v001` ne doit donc pas encore être figée comme vérité terrain.**

Nous ne corrigeons pas automatiquement ces images : une règle telle que « supprimer toutes les grandes boîtes » retirerait aussi de vrais gros plans. Nous construisons plutôt `qa_review_queue.csv` avec des signaux explicables :

- `large_bbox` : diamètre équivalent d'au moins 32 px à 640 ;
- `single_negative_gap` : une frame négative isolée entre deux positives ;
- `center_jump` : déplacement supérieur à 25 % de la diagonale normalisée en une frame ;
- `area_jump` : aire multipliée ou divisée par plus de quatre en une frame ;
- `visual_sample_*` : erreur déjà confirmée dans les galeries ci-dessus.

Ces règles servent uniquement à **prioriser la revue humaine**. Une image absente de la file n'est pas garantie parfaite, et une image présente n'est pas forcément fausse.

In [ ]:
QA_REVIEW_QUEUE_PATH = PROCESSED_ROOT / "qa_review_queue.csv"

# Cinq erreurs certaines observées dans l'échantillon. Les autres cas restent à décider.
confirmed_visual_findings = {
    "48797f76-4da7-43a7-9d52-9a7cdf9042ef_000015": (
        "visual_sample_false_positive", "boîte sur un crâne ; ballon absent"
    ),
    "143e0618-a3e8-448c-a133-42f97bca8db2_000006": (
        "visual_sample_false_positive", "boîte dans la foule ; ballon absent"
    ),
    "5ee600ae-18ce-4307-82e4-8458b2767ee2_000017": (
        "visual_sample_false_positive", "boîte sur une zone lumineuse ; ballon absent"
    ),
    "81926caa-6cb9-4206-b709-60d55623d18a_000027": (
        "visual_sample_false_positive", "boîte au bord gauche ; ballon absent"
    ),
    "d52fe89a-6572-4129-b726-20eec92ef146_000284": (
        "visual_sample_false_negative", "gardien tenant un ballon ; label vide"
    ),
}

for record in accepted_records:
    record["review_reasons"] = set()
    record["human_finding"] = ""
    if record["diameter_640"] is not None and record["diameter_640"] >= 32:
        record["review_reasons"].add("large_bbox")
    manual = confirmed_visual_findings.get(record["image_path"].stem)
    if manual:
        reason, finding = manual
        record["review_reasons"].add(reason)
        record["human_finding"] = finding

accepted_by_sequence = defaultdict(list)
for record in accepted_records:
    accepted_by_sequence[record["sequence"]].append(record)

for sequence_records in accepted_by_sequence.values():
    by_frame = {record["frame"]: record for record in sequence_records}
    for record in sequence_records:
        frame = record["frame"]
        previous_record = by_frame.get(frame - 1)
        next_record = by_frame.get(frame + 1)
        if (
            record["status"] == "negative"
            and previous_record is not None
            and next_record is not None
            and previous_record["status"] == "positive_single"
            and next_record["status"] == "positive_single"
        ):
            record["review_reasons"].add("single_negative_gap")

        if (
            record["status"] == "positive_single"
            and previous_record is not None
            and previous_record["status"] == "positive_single"
        ):
            x, y, width, height = record["boxes"][0]
            previous_x, previous_y, previous_width, previous_height = (
                previous_record["boxes"][0]
            )
            center_distance = math.hypot(x - previous_x, y - previous_y)
            area_ratio = max(
                (width * height) / (previous_width * previous_height),
                (previous_width * previous_height) / (width * height),
            )
            if center_distance > 0.25:
                record["review_reasons"].add("center_jump")
            if area_ratio > 4:
                record["review_reasons"].add("area_jump")

review_candidates = [
    record for record in accepted_records if record["review_reasons"]
]
review_fields = [
    "sequence", "frame", "split", "current_status",
    "diameter_640", "review_reasons", "human_finding",
    "source_image", "source_label",
]
with QA_REVIEW_QUEUE_PATH.open("w", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=review_fields)
    writer.writeheader()
    for record in sorted(
        review_candidates, key=lambda item: (item["split"], item["sequence"], item["frame"])
    ):
        writer.writerow({
            "sequence": record["sequence"],
            "frame": record["frame"],
            "split": record["split"],
            "current_status": record["status"],
            "diameter_640": (
                f"{record['diameter_640']:.3f}"
                if record["diameter_640"] is not None else ""
            ),
            "review_reasons": ";".join(sorted(record["review_reasons"])),
            "human_finding": record["human_finding"],
            "source_image": relative_to_project(record["image_path"]),
            "source_label": (
                relative_to_project(record["label_path"])
                if record["label_path"].exists() else ""
            ),
        })

reason_counts = Counter(
    reason for record in review_candidates for reason in record["review_reasons"]
)
split_review_counts = Counter(record["split"] for record in review_candidates)
display(Markdown(f"""
### File de correction créée

- **{len(review_candidates):,} images prioritaires** sur {accepted_count:,} ;
- train : **{split_review_counts['train']:,}** ;
- validation : **{split_review_counts['val']:,}** ;
- test : **{split_review_counts['test']:,}** ;
- erreurs déjà confirmées visuellement : **{len(confirmed_visual_findings)}**.

Le CSV indique pourquoi chaque frame a été sélectionnée. Il ne modifie ni `raw` ni les labels de `v001`.
"""))

## 8. Contrôle indépendant des doublons

La séparation par UUID garantit qu'une séquence ne traverse pas les splits. Nous calculons en plus le SHA-256 des 7 014 images préparées pour vérifier qu'aucun fichier identique n'est présent sous un autre nom dans un autre split. Ce contrôle détecte les doublons exacts, pas deux frames différentes mais visuellement très proches.

In [ ]:
import hashlib

hash_groups = defaultdict(list)
for record in accepted_records:
    prepared_image = (
        PROCESSED_ROOT / record["split"] / "images" / record["image_path"].name
    )
    digest = hashlib.sha256(prepared_image.read_bytes()).hexdigest()
    hash_groups[digest].append(record)

exact_duplicate_groups = [group for group in hash_groups.values() if len(group) > 1]
cross_split_duplicate_groups = [
    group
    for group in exact_duplicate_groups
    if len({record["split"] for record in group}) > 1
]

assert len(hash_groups) == accepted_count
assert not exact_duplicate_groups
assert not cross_split_duplicate_groups

qa_report = {
    "dataset_id": "pitchlens_infactory_v001",
    "source": "Infactory Ball Tracking",
    "visual_sampling_seed": 42,
    "visual_samples_per_split": len(next(iter(qa_samples.values()))),
    "visual_samples_total": sum(len(sample) for sample in qa_samples.values()),
    "visual_categories": list(QA_CATEGORY_ORDER),
    "population_by_split": {
        split: dict(qa_population[split]) for split in ("train", "val", "test")
    },
    "unique_image_hashes": len(hash_groups),
    "exact_duplicate_groups": len(exact_duplicate_groups),
    "cross_split_duplicate_groups": len(cross_split_duplicate_groups),
    "mechanical_validation": "passed",
    "human_visual_validation": "failed_sample_contains_label_errors",
    "confirmed_visual_errors": len(confirmed_visual_findings),
    "priority_review_images": len(review_candidates),
    "priority_review_by_split": dict(split_review_counts),
    "priority_review_reasons": dict(reason_counts),
}
QA_REPORT_PATH.write_text(json.dumps(qa_report, indent=2) + "\n")

display(Markdown(f"""
### Contrôle des doublons réussi

- **{len(hash_groups):,} empreintes uniques** pour {accepted_count:,} images ;
- **{len(exact_duplicate_groups)} doublon exact** dans tout le dataset ;
- **{len(cross_split_duplicate_groups)} doublon exact** entre les splits.

Le contrôle mécanique est terminé, mais la validation visuelle a échoué : `v001` reste une version de travail jusqu'au traitement de la file de correction.
"""))

## Résultat

`v001` est une baseline **mécaniquement cohérente**, protégée contre la fuite entre séquences et contrôlée contre les doublons exacts. En revanche, l'échantillon visuel contient des erreurs sémantiques : cette version reste une version de travail et ne doit pas encore être considérée comme une vérité terrain propre.

La prochaine action data est de traiter en priorité `qa_review_queue.csv`, en commençant par les cas de validation/test, puis de produire une `v002` sans jamais écraser `raw` ni `v001`. Les 138 annotations invalides ou multi-boîtes de la première file de revue rejoindront cette correction.

## 9. Construire v002 après la revue humaine

Règle décidée ensemble : **sans décision = conserver le label d'origine**, y compris plusieurs vrais ballons. `keep` conserve, `negative` vide le label, `bbox` remplace les boîtes par la correction et `exclude` retire l'image. Les exclusions par séquence sont déjà enregistrées comme décisions.

Le constructeur relit le manifeste complet (7 152 images), pas seulement les fichiers retenus dans v001. Il conserve les splits par séquence, copie les fichiers dans v002 et fige une copie des décisions avec leurs empreintes SHA-256. Les annotations invalides non corrigées bloquent la construction. Une version existante n'est jamais écrasée.

Le code réutilisable dans `src/pitchlens/data/build_v002.py` assure la même construction depuis ce notebook ou une commande. Le rapport distingue les annotations acceptées par défaut des décisions explicites ; v002 n'est pas présentée comme entièrement corrigée image par image.

In [ ]:
from pitchlens.data.build_v002 import build

v002_root = PROJECT_ROOT / 'data/processed/v002'
if v002_root.exists():
    v002_report = json.loads((v002_root / 'build_report.json').read_text())
    print('v002 existe déjà : lecture du rapport figé.')
else:
    v002_report = build(PROJECT_ROOT)
display({key: v002_report[key] for key in ('images', 'actions', 'splits', 'quality_status')})